# Experimente — Clasificare Status Final

Notebook pentru rularea și evaluarea prompturilor de clasificare a statusului final al conversațiilor.

**Task:** Clasifică statusul final al conversației în una dintre cele 5 categorii.

**Output model:**
```json
{
  "final_status": "<status>",
  "confidence": "high|medium|low",
  "reasoning": "<justificare>"
}
```

**Statusuri finale:** `resolved`, `escalated`, `partial`, `failed`, `out_of_scope`

**Structură celule:**
- [0] Setup
- [1] Config experiment
- [2] Prompt renderer + funcții utilitare
- [3] Init model
- [4] Test prompt
- [5] Rulare completă
- [6] Metrici și rezultate
- [7] Salvare experiment
- [8] Comparație experimente

In [1]:
# [0] Setup — paths și imports
import os

os.chdir(
    r"C:\Users\Matebook 14s\Documents"
    r"\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-"
)

print("Working directory:", os.getcwd())

Working directory: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-


In [2]:
# [0b] Imports și paths
import json
import time
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
from jinja2 import Environment, FileSystemLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report
from tabulate import tabulate

# ── Paths ──────────────────────────────────────────────────────────────────
PROMPTS_DIR  = Path("prompts/final_status")
CONFIGS_DIR  = Path("configs")
DATASET_PATH = Path("data/master_dataset_refined_180.json")
OUTPUT_DIR   = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ─────────────────────────────────────────────────────────────────
with open(CONFIGS_DIR / "final_status_definitions.json", encoding="utf-8") as f:
    _defs = json.load(f)
STATUS_LABELS = [l["name"] for l in _defs["labels"]]
VALID_STATUSES = set(STATUS_LABELS)

# ── Dataset ────────────────────────────────────────────────────────────────
with open(DATASET_PATH, encoding="utf-8") as f:
    DATASET = json.load(f)["conversations"]

# Statistici dataset
status_counts = defaultdict(int)
for c in DATASET:
    status_counts[c["final_status"]] += 1

print(f"✓ Dataset: {len(DATASET)} conversații")
print(f"✓ Statusuri: {STATUS_LABELS}")
print(f"  Distribuție:")
for status in STATUS_LABELS:
    count = status_counts.get(status, 0)
    print(f"    {status:15s}: {count:3d} ({100*count/len(DATASET):.1f}%)")
print(f"✓ Output: {OUTPUT_DIR}")

✓ Dataset: 180 conversații
✓ Statusuri: ['rezolvata', 'nerezolvata', 'redirectionata', 'partial_rezolvata', 'intrerupta']
  Distribuție:
    rezolvata      :  85 (47.2%)
    nerezolvata    :  13 (7.2%)
    redirectionata :  28 (15.6%)
    partial_rezolvata:  34 (18.9%)
    intrerupta     :  20 (11.1%)
✓ Output: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status


In [ ]:
# [1] Configurație experiment — editează aici
# ═══════════════════════════════════════════════════════════

MODEL          = "gemini_2.5_flash"       # openai_o3 | gemini_2.5_flash | aya_expanse_8b | rollama2_7b | roberta_encoder
LANG           = "en"              # ro | en
PROMPT_VERSION = "v4"              # v1 | v2 | v3 | v4

# Rulează pe tot datasetul sau doar pe primele N conversații (None = tot)
MAX_CONVERSATIONS = None

# ═══════════════════════════════════════════════════════════
EXP_NAME = f"fst_{MODEL}__{LANG}__{PROMPT_VERSION}"
EXP_FILE = OUTPUT_DIR / f"exp_{EXP_NAME}.json"

conversations = DATASET[:MAX_CONVERSATIONS] if MAX_CONVERSATIONS else DATASET

print(f"Experiment : {EXP_NAME}")
print(f"Model      : {MODEL}")
print(f"Limbă      : {LANG}")
print(f"Prompt     : {LANG}_final_status_{PROMPT_VERSION}.jinja")
print(f"Dataset    : {len(conversations)} conversații")
print(f"Output     : {EXP_FILE}")

Experiment : fst_gemini_2.5_flash__ro__v4
Model      : gemini_2.5_flash
Limbă      : en
Prompt     : en_final_status_v4.jinja
Dataset    : 180 conversații
Output     : C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status\exp_fst_gemini_2.5_flash__ro__v4.json


In [4]:
# [2] Prompt renderer + parser + funcții utilitare
# ─────────────────────────────────────────────────────────────────────────

env = Environment(loader=FileSystemLoader(str(PROMPTS_DIR)))

def render_prompt(lang: str, turns: list, version: str = "v1", few_shot_examples: list = None) -> str:
    """
    Renderizează promptul Jinja2 cu conversația și opțional cu few-shot examples.
    
    Args:
        lang: 'ro' sau 'en'
        turns: lista de turn-uri
        version: 'v1', 'v2', 'v3', 'v4'
        few_shot_examples: lista de exemple pentru few-shot (doar pentru v4)
    """
    template = env.get_template(f"{lang}_final_status_{version}.jinja")
    
    if few_shot_examples:
        return template.render(conversation=turns, examples=few_shot_examples)
    else:
        return template.render(conversation=turns)

# ── Response parser ────────────────────────────────────────────────────────
def parse_response(raw: str):
    """
    Returnează (final_status, confidence, reasoning, parse_failed).
    Gestionează JSON înfășurat în ```json ... ```.
    """
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        data = json.loads(text)
        status = data.get("final_status")
        # Validare
        if status not in VALID_STATUSES:
            return status, data.get("confidence"), data.get("reasoning"), True
        return (
            status,
            data.get("confidence"),
            data.get("reasoning"),
            False,
        )
    except json.JSONDecodeError:
        return None, None, None, True

# ── Helper: eticheta din dataset ──────────────────────────────────────────
def get_dataset_label(conv: dict):
    """Returnează final_status din dataset."""
    return conv.get("final_status")

# ── Metrici ────────────────────────────────────────────────────────────────
def compute_metrics(results: list):
    """Metrici pentru clasificarea statusului final (5 clase)."""
    valid = [r for r in results if not r["parse_failed"] and r["predicted_status"] is not None]
    if not valid:
        return {}
    y_true = [r["dataset_status"] for r in valid]
    y_pred = [r["predicted_status"] for r in valid]
    return {
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":    round(f1_score(y_true, y_pred, average="macro", labels=STATUS_LABELS, zero_division=0), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", labels=STATUS_LABELS, zero_division=0), 4),
        "kappa":       round(cohen_kappa_score(y_true, y_pred), 4),
        "n_valid":     len(valid),
    }

# ── Tabel metrici sumar ────────────────────────────────────────────────────
def print_metrics_summary(results: list, exp_name: str):
    fails = sum(1 for r in results if r["parse_failed"])
    total = len(results)
    lats  = [r["latency_ms"] for r in results if r["latency_ms"] > 0]

    m = compute_metrics(results)

    rows = [
        ["Experiment",        exp_name],
        ["Accuracy",          f"{m.get('accuracy', 0)*100:.1f}%"],
        ["Macro F1",          f"{m.get('macro_f1', 0)*100:.1f}%"],
        ["Weighted F1",       f"{m.get('weighted_f1', 0)*100:.1f}%"],
        ["Cohen's Kappa",     f"{m.get('kappa', 0):.3f}"],
        ["Parse failures",    f"{fails}/{total}  ({fails/total*100:.1f}%)"],
        ["Latență medie",     f"{np.mean(lats):.0f}ms" if lats else "—"],
        ["Latență p95",       f"{np.percentile(lats, 95):.0f}ms" if lats else "—"],
        ["N valid",           m.get('n_valid', 0)],
    ]
    print(tabulate(rows, tablefmt="simple", headers=["Metrică", "Valoare"]))
    return {**m, "n_failed": fails}

# ── Tabel predicții ────────────────────────────────────────────────────────
def print_predictions_table(results: list):
    rows = []
    for r in results:
        if r["parse_failed"]:
            status = "FAIL"
        elif r["predicted_status"] == r["dataset_status"]:
            status = f"✓  {r['predicted_status']}"
        else:
            status = f"✗  {r['predicted_status']}  (real: {r['dataset_status']})"
        rows.append([
            r["conversation_id"],
            r["dataset_status"],
            status,
            r.get("confidence", "—"),
            f"{r['latency_ms']:.0f}ms",
        ])
    print(tabulate(rows,
                   headers=["conv_id", "dataset_label", "predicție", "confidence", "latență"],
                   tablefmt="rounded_outline"))

# ── Top erori ──────────────────────────────────────────────────────────────
def print_top_errors(results: list):
    errors = defaultdict(lambda: defaultdict(int))
    for r in results:
        if r["parse_failed"]:
            continue
        true_status = r["dataset_status"]
        pred_status = r["predicted_status"]
        if true_status != pred_status:
            errors[true_status][pred_status] += 1

    rows = [
        [true, pred, cnt]
        for true, preds in errors.items()
        for pred, cnt in preds.items()
    ]
    rows.sort(key=lambda r: -r[2])
    if rows:
        print(tabulate(rows[:10],
                       headers=["Status real", "Prezis ca", "# greșeli"],
                       tablefmt="simple"))
    else:
        print("  0 erori.")

# ── Classification report ─────────────────────────────────────────────────
def print_classification_report(results: list):
    valid = [r for r in results if not r["parse_failed"] and r["predicted_status"] is not None]
    if not valid:
        print("Nu există predicții valide pentru raport.")
        return
    y_true = [r["dataset_status"] for r in valid]
    y_pred = [r["predicted_status"] for r in valid]
    print(classification_report(y_true, y_pred, labels=STATUS_LABELS, zero_division=0))

print("✓ Funcții utilitare încărcate.")

✓ Funcții utilitare încărcate.


In [5]:
# [3] Inițializează modelul selectat în [1]
# ─────────────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

if MODEL == "openai_o3":
    from openai import OpenAI
    _client = OpenAI(max_retries=5)

    def call_model(prompt: str) -> str:
        r = _client.chat.completions.create(
            model="o3",
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content

elif MODEL == "gemini_2.5_flash":
    import os
    from google.genai import Client
    _client = Client(api_key=os.environ["GEMINI_API_KEY"])

    def call_model(prompt: str) -> str:
        response = _client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text

elif MODEL == "aya_expanse_8b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="aya-expanse:8b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "rollama2_7b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="rollama2-instruct:7b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "roberta_encoder":
    # XLM-RoBERTa XNLI — zero-shot classification NLI
    from transformers import pipeline as hf_pipeline
    _pipe = hf_pipeline(
        "zero-shot-classification",
        model="joeddav/xlm-roberta-large-xnli",
        device=-1,
    )

    def _extract_conversation_text(prompt: str) -> str:
        lines = prompt.split("\n")
        conv_lines, in_conv = [], False
        for line in lines:
            stripped = line.strip()
            if "Convers" in stripped and stripped.startswith("#"):
                in_conv = True
                continue
            if in_conv and stripped.startswith("#"):
                break
            if in_conv and stripped:
                conv_lines.append(stripped)
        return " ".join(conv_lines)

    def call_model(prompt: str) -> str:
        text = _extract_conversation_text(prompt)
        candidate_labels = STATUS_LABELS  # ["resolved", "escalated", "partial", "failed", "out_of_scope"]
        result = _pipe(text, candidate_labels=candidate_labels)
        predicted = result["labels"][0]
        score     = result["scores"][0]
        conf = "high" if score > 0.7 else "medium" if score > 0.4 else "low"
        return json.dumps({
            "final_status": predicted,
            "confidence":   conf,
            "reasoning":    f"zero-shot score: {score:.4f}",
        })
else:
    raise ValueError(f"Model necunoscut: {MODEL}")

print(f"✓ Model inițializat: {MODEL}")

✓ Model inițializat: gemini_2.5_flash


In [6]:
# [4] Test prompt pe o conversație
# ─────────────────────────────────────────────────────────────────────────

# Încarcă few-shot examples pentru v4
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_file = "few_shot_examples_final_status.json"
    try:
        with open(CONFIGS_DIR / few_shot_file, encoding="utf-8") as f:
            few_shot_data = json.load(f)
            few_shot_examples = few_shot_data.get("examples", [])
        print(f"✓ Încărcat {len(few_shot_examples)} exemple few-shot pentru {LANG} {PROMPT_VERSION}")
    except FileNotFoundError:
        print(f"⚠ Nu s-a găsit {few_shot_file} - se continuă fără few-shot examples")

sample_conv = conversations[0]
ds_status = get_dataset_label(sample_conv)

sample_prompt = render_prompt(LANG, sample_conv["turns"], PROMPT_VERSION, few_shot_examples)

print(f"── Conversație: {sample_conv['conversation_id']}")
print(f"── dataset final_status: {ds_status}")
print("─" * 60)
print(sample_prompt)
print("─" * 60)

raw = call_model(sample_prompt)
print("RAW:", repr(raw))
parsed_status, conf, reason, failed = parse_response(raw)
print(f"PARSED: status={parsed_status}, confidence={conf}, failed={failed}")
print(f"REASONING: {reason}")

✓ Încărcat 7 exemple few-shot pentru en v4
── Conversație: conv_simple_0001
── dataset final_status: rezolvata
────────────────────────────────────────────────────────────
Developer: You are a conversation outcome classifier for a Romanian banking voicebot system.

Your task is to classify the **final status** of the conversation below, based on how it ended and whether the user's request was fulfilled.

## Status labels

**rezolvata**
The voicebot completed everything it could do within the conversation for the user's request. This includes: registering a report, blocking a card, sending a statement, providing requested information, activating a service, scheduling a meeting, or other banking requests. The fact that the user must **wait** for a later result (investigation, card delivery, processing) does not make the conversation partially resolved — if the bot executed the requested action and has nothing left to do, the status is `rezolvata`.

**partial_rezolvata**
The request was p

In [9]:
# [5] Rulare completă pe dataset
# ─────────────────────────────────────────────────────────────────────────

# Încarcă few-shot examples pentru v4
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_file = "few_shot_examples_final_status.json" 
    try:
        with open(CONFIGS_DIR / few_shot_file, encoding="utf-8") as f:
            few_shot_data = json.load(f)
            few_shot_examples = few_shot_data.get("examples", [])
        print(f"✓ Încărcat {len(few_shot_examples)} exemple few-shot")
    except FileNotFoundError:
        print(f"⚠ {few_shot_file} nu a fost găsit - se continuă fără few-shot examples")

results = []
total = len(conversations)

print(f"Rulare experiment: {EXP_NAME}")
print(f"Dataset: {total} conversații\n")

for i, conv in enumerate(conversations):
    conv_id   = conv["conversation_id"]
    ds_status = get_dataset_label(conv)
    turns     = conv["turns"]

    prompt = render_prompt(LANG, turns, PROMPT_VERSION, few_shot_examples)

    start = time.perf_counter()
    try:
        raw = call_model(prompt)
        pred_status, confidence, reasoning, parse_failed = parse_response(raw)
        elapsed_ms = (time.perf_counter() - start) * 1000

        results.append({
            "conversation_id": conv_id,
            "dataset_status": ds_status,
            "predicted_status": pred_status,
            "confidence": confidence,
            "reasoning": reasoning,
            "raw_response": raw,
            "parse_failed": parse_failed,
            "latency_ms": elapsed_ms,
        })

        # Progress indicator
        if parse_failed:
            print(f"[{i+1:3d}/{total}] ✗ {conv_id} — PARSE FAILED")
        elif pred_status == ds_status:
            print(f"[{i+1:3d}/{total}] ✓ {conv_id}")
        else:
            print(f"[{i+1:3d}/{total}] ✗ {conv_id} — pred:{pred_status} vs real:{ds_status}")

    except Exception as e:
        elapsed_ms = (time.perf_counter() - start) * 1000
        results.append({
            "conversation_id": conv_id,
            "dataset_status": ds_status,
            "predicted_status": None,
            "confidence": None,
            "reasoning": None,
            "raw_response": "",
            "parse_failed": True,
            "latency_ms": elapsed_ms,
        })
        print(f"[{i+1:3d}/{total}] ✗ {conv_id} — EXCEPTION: {e}")

    # Batch progress
    if (i + 1) % 20 == 0:
        print(f"\n── Progres: {i+1}/{total} ({100*(i+1)/total:.1f}%) ──\n")

print(f"\n✓ Experiment finalizat: {len(results)} conversații procesate")

✓ Încărcat 7 exemple few-shot
Rulare experiment: fst_gemini_2.5_flash__ro__v4
Dataset: 180 conversații

[  1/180] ✓ conv_simple_0001
[  2/180] ✓ conv_simple_0002
[  3/180] ✓ conv_simple_0003
[  4/180] ✓ conv_simple_0004
[  5/180] ✓ conv_simple_0005
[  6/180] ✓ conv_simple_0006
[  7/180] ✓ conv_simple_0007
[  8/180] ✓ conv_simple_0008
[  9/180] ✓ conv_simple_0009
[ 10/180] ✓ conv_simple_0010
[ 11/180] ✓ conv_simple_0011
[ 12/180] ✓ conv_simple_0012
[ 13/180] ✓ conv_simple_0013
[ 14/180] ✓ conv_simple_0014
[ 15/180] ✓ conv_simple_0015
[ 16/180] ✓ conv_simple_0016
[ 17/180] ✓ conv_simple_0017
[ 18/180] ✓ conv_simple_0018
[ 19/180] ✓ conv_simple_0019
[ 20/180] ✓ conv_simple_0020

── Progres: 20/180 (11.1%) ──

[ 21/180] ✓ conv_simple_0021
[ 22/180] ✓ conv_simple_0022
[ 23/180] ✓ conv_simple_0023
[ 24/180] ✓ conv_simple_0024
[ 25/180] ✓ conv_simple_0025
[ 26/180] ✓ conv_simple_0026
[ 27/180] ✓ conv_simple_0027
[ 28/180] ✓ conv_simple_0028
[ 29/180] ✓ conv_simple_0029
[ 30/180] ✓ conv_simple

In [10]:
# [6] Metrici și analiză rezultate
# ─────────────────────────────────────────────────────────────────────────

print("═" * 80)
print(f"REZULTATE EXPERIMENT: {EXP_NAME}")
print("═" * 80)
print()

# Metrici sumar
print("📊 METRICI SUMAR")
print("─" * 80)
metrics = print_metrics_summary(results, EXP_NAME)
print()

# Classification report
print("📋 CLASSIFICATION REPORT")
print("─" * 80)
print_classification_report(results)
print()

# Top erori
print("❌ TOP ERORI DE CLASIFICARE")
print("─" * 80)
print_top_errors(results)
print()

# Tabel predicții (primele 20)
print("📝 PREDICȚII (primele 20)")
print("─" * 80)
print_predictions_table(results[:20])
print()

print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
REZULTATE EXPERIMENT: fst_gemini_2.5_flash__ro__v4
════════════════════════════════════════════════════════════════════════════════

📊 METRICI SUMAR
────────────────────────────────────────────────────────────────────────────────
Metrică         Valoare
--------------  ----------------------------
Experiment      fst_gemini_2.5_flash__ro__v4
Accuracy        86.1%
Macro F1        83.4%
Weighted F1     86.3%
Cohen's Kappa   0.804
Parse failures  0/180  (0.0%)
Latență medie   4769ms
Latență p95     11370ms
N valid         180

📋 CLASSIFICATION REPORT
────────────────────────────────────────────────────────────────────────────────
                   precision    recall  f1-score   support

        rezolvata       0.95      0.89      0.92        85
      nerezolvata       0.71      0.92      0.80        13
   redirectionata       0.86      0.89      0.88        28
partial_rezolvata       0.73      0.79      0.7

In [ ]:
# [7] Salvare rezultate experiment
# ─────────────────────────────────────────────────────────────────────────

experiment_data = {
    "experiment_name": EXP_NAME,
    "model": MODEL,
    "language": LANG,
    "prompt_version": PROMPT_VERSION,
    "dataset_size": len(conversations),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": compute_metrics(results),
    "results": results,
}

with open(EXP_FILE, "w", encoding="utf-8") as f:
    json.dump(experiment_data, f, ensure_ascii=False, indent=2)

print(f"✓ Experiment salvat: {EXP_FILE}")
print(f"  Model: {MODEL}")
print(f"  Limbă: {LANG}")
print(f"  Prompt: {PROMPT_VERSION}")
print(f"  Accuracy: {metrics['accuracy']*100:.1f}%")
print(f"  Macro F1: {metrics['macro_f1']*100:.1f}%")

✓ Experiment salvat: C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status\exp_fst_gemini_2.5_flash__ro__v4.json
  Model: gemini_2.5_flash
  Limbă: en
  Prompt: v4
  Accuracy: 86.1%
  Macro F1: 83.4%


In [10]:
# [8] Comparație experimente
# ─────────────────────────────────────────────────────────────────────────

import json
from pathlib import Path
from tabulate import tabulate

# Dacă OUTPUT_DIR nu e deja definit, decomentează linia de mai jos:
# OUTPUT_DIR = Path("outputs_final_status")

experiment_files = sorted(OUTPUT_DIR.glob("exp_fst_*.json"))

if not experiment_files:
    print("Nu există experimente salvate.")
else:
    comparisons = []

    for exp_file in experiment_files:
        with open(exp_file, "r", encoding="utf-8") as f:
            exp_data = json.load(f)

        # Compatibilitate cu structuri diferite
        experiment_name = exp_data.get(
            "experiment_name",
            exp_data.get("experiment", {}).get("name", exp_file.stem)
        )

        model = exp_data.get("model", "unknown")
        language = exp_data.get("language", "unknown")
        prompt_version = exp_data.get("prompt_version", "unknown")

        metrics = exp_data.get("metrics", {})

        accuracy = metrics.get("accuracy", 0)
        macro_f1 = metrics.get("macro_f1", 0)
        weighted_f1 = metrics.get("weighted_f1", 0)
        kappa = metrics.get("cohen_kappa", 0)

        # Suportă atât "results", cât și "predictions"
        preds = exp_data.get("results", exp_data.get("predictions", []))

        n_total = len(preds)
        n_failed = metrics.get(
            "n_failed",
            sum(1 for p in preds if p.get("parse_failed", False))
        )

        parse_fail_pct = (n_failed / n_total * 100) if n_total else 0

        comparisons.append({
            "experiment_name": experiment_name,
            "model": model,
            "language": language,
            "prompt_version": prompt_version,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "kappa": kappa,
            "n_total": n_total,
            "n_failed": n_failed,
            "parse_fail_pct": parse_fail_pct,
            "file": exp_file.name,
        })

    # Sortează după Macro F1 descrescător
    comparisons = sorted(comparisons, key=lambda x: x["macro_f1"], reverse=True)

    table = []

    for rank, exp in enumerate(comparisons, start=1):
        best_marker = "★" if rank == 1 else ""

        table.append([
            rank,
            best_marker,
            exp["experiment_name"],
            exp["model"],
            exp["language"],
            exp["prompt_version"],
            f"{exp['accuracy'] * 100:.1f}%",
            f"{exp['macro_f1'] * 100:.1f}%",
            f"{exp['weighted_f1'] * 100:.1f}%",
            f"{exp['kappa']:.3f}",
            exp["n_total"],
            exp["n_failed"],
            f"{exp['parse_fail_pct']:.1f}%",
        ])

    print("═" * 140)
    print("COMPARAȚIE EXPERIMENTE FINAL_STATUS")
    print("═" * 140)

    print(tabulate(
        table,
        headers=[
            "Rank",
            "Best",
            "Experiment",
            "Model",
            "Lang",
            "Ver",
            "Acc",
            "Macro F1",
            "Weighted F1",
            "Kappa",
            "N",
            "Fails",
            "Fail %",
        ],
        tablefmt="rounded_outline"
    ))

    print("═" * 140)

    best = comparisons[0]

    print("\nBEST EXPERIMENT:")
    print(f"  Experiment: {best['experiment_name']}")
    print(f"  Model:      {best['model']}")
    print(f"  Language:   {best['language']}")
    print(f"  Version:    {best['prompt_version']}")
    print(f"  Accuracy:   {best['accuracy'] * 100:.1f}%")
    print(f"  Macro F1:   {best['macro_f1'] * 100:.1f}%")
    print(f"  Weighted F1:{best['weighted_f1'] * 100:.1f}%")
    print(f"  Kappa:      {best['kappa']:.3f}")
    print(f"  Fails:      {best['n_failed']}/{best['n_total']} ({best['parse_fail_pct']:.1f}%)")

════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
COMPARAȚIE EXPERIMENTE FINAL_STATUS
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
╭────────┬────────┬──────────────────────────────┬──────────────────┬─────────┬─────────┬───────┬────────────┬───────────────┬─────────┬─────┬─────────┬──────────╮
│   Rank │ Best   │ Experiment                   │ Model            │ Lang    │ Ver     │ Acc   │ Macro F1   │ Weighted F1   │   Kappa │   N │   Fails │ Fail %   │
├────────┼────────┼──────────────────────────────┼──────────────────┼─────────┼─────────┼───────┼────────────┼───────────────┼─────────┼─────┼─────────┼──────────┤
│      1 │ ★      │ fst_gemini_2.5_flash__ro__v3 │ gemini_2.5_flash │ ro      │ v3      │ 84.4% │ 79.3%      │ 84.2%         │       0 │ 180 │       1 │ 0.6%     │
│      2 │        │ fst_ge

In [13]:
PREDS_KEY = LABEL_FIELD = PRED_FIELD = CONV_FIELD = None

In [11]:
# =============================================================
# AGREGARE REZULTATE FINALE — fst v1/v2/v3
# Scopul: identificarea erorilor sistematice pentru a construi
# exemplele few-shot pentru v4
# =============================================================

import json, os, glob
from collections import Counter, defaultdict

OUTPUTS_DIR  = "outputs_final_status"
VALID_LABELS = ["rezolvata", "partial_rezolvata", "redirectionata", "intrerupta", "nerezolvata"]

# Field names confirmate din structura ta
PREDS_KEY   = "results"
LABEL_FIELD = "dataset_status"
PRED_FIELD  = "predicted_status"
CONV_FIELD  = "conversation_id"

# --- 1. ÎNCARCĂ TOATE EXPERIMENTELE fst_ ---
experiments = {}
for fpath in sorted(glob.glob(os.path.join(OUTPUTS_DIR, "*fst_*.json"))):
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    name = data.get("experiment_name",
           data.get("experiment", {}).get("name",
           os.path.basename(fpath).replace(".json", "")))
    experiments[name] = data

print(f"Experimente găsite: {len(experiments)}")
for name in sorted(experiments):
    print(f"  • {name}")

# detectează cheia pentru kappa din primul experiment
sample_metrics = list(experiments.values())[0].get("metrics", {})
kappa_key = next((k for k in sample_metrics if "kappa" in k.lower()), None)
print(f"\nChei metrics: {list(sample_metrics.keys())}")
print(f"Kappa key: '{kappa_key}'\n")

# --- 2. TABEL SUMAR TOATE VERSIUNILE ---
print(f"{'='*90}")
print(f"SUMAR METRICI — toate versiunile")
print(f"{'='*90}")
print(f"{'Experiment':<50} {'Acc':>6} {'MacF1':>6} {'WgtF1':>6} {'Kappa':>7}")
print(f"{'-'*90}")

results_table = []
for name, data in sorted(experiments.items()):
    m    = data.get("metrics", {})
    acc  = m.get("accuracy", 0)
    mf1  = m.get("macro_f1", 0)
    wf1  = m.get("weighted_f1", 0)
    kap  = m.get(kappa_key, 0) if kappa_key else 0
    results_table.append((name, acc, mf1, wf1, kap))
    print(f"{name:<50} {acc:>6.3f} {mf1:>6.3f} {wf1:>6.3f} {kap:>7.3f}")

best = max(results_table, key=lambda x: x[2])
print(f"\n★  Best Macro-F1: {best[0]}  ({best[2]:.3f})")

# --- 3. BEST PER VERSIUNE ---
print(f"\n{'='*90}")
print(f"BEST CONFIG PER VERSIUNE (Macro-F1)")
print(f"{'='*90}")
for version in ["v1", "v2", "v3"]:
    subset = [r for r in results_table if r[0].endswith(version) or f"__{version}" in r[0] or f"_{version}" in r[0]]
    if subset:
        b = max(subset, key=lambda x: x[2])
        print(f"  {version}: {b[0]:<48} Macro-F1={b[2]:.3f}  Kappa={b[4]:.3f}")

# --- 4. ANALIZA ERORILOR PE BEST CONFIG ---
best_name = best[0]
best_data = experiments[best_name]
preds     = best_data[PREDS_KEY]

print(f"\n{'='*90}")
print(f"ANALIZA ERORILOR — {best_name}")
print(f"{'='*90}")

errors           = defaultdict(list)
correct_by_class = defaultdict(list)

for p in preds:
    true = p.get(LABEL_FIELD, "?")
    pred = p.get(PRED_FIELD, "PARSE_FAIL")
    if p.get("parse_failed", False):
        pred = "PARSE_FAIL"
    if true == pred:
        correct_by_class[true].append(p)
    else:
        errors[(true, pred)].append(p)

total_errors = sum(len(v) for v in errors.values())
total        = len(preds)
print(f"\nTotal erori: {total_errors}/{total}  ({total_errors/total*100:.1f}%)\n")

print(f"{'Confuzie (true → pred)':<50} {'N':>4}  {'%':>6}")
print(f"{'-'*65}")
for (true, pred), lst in sorted(errors.items(), key=lambda x: -len(x[1])):
    bar = "█" * len(lst)
    print(f"  {true:<25} → {pred:<20} {len(lst):>3}   {len(lst)/total*100:>5.1f}%  {bar}")

# FN per clasă
print(f"\n{'─'*65}")
print(f"FALSE NEGATIVES per clasă:")
print(f"{'─'*65}")
for label in VALID_LABELS:
    fn         = sum(len(v) for (t, p), v in errors.items() if t == label)
    total_true = sum(1 for p in preds if p.get(LABEL_FIELD) == label)
    print(f"  {label:<28} FN={fn:>2}/{total_true:<3} ", end="")
    breakdown = Counter({p: len(v) for (t, p), v in errors.items() if t == label})
    for pred_as, cnt in breakdown.most_common(3):
        print(f"  →{pred_as}:{cnt}", end="")
    print()

# FP per clasă
print(f"\n{'─'*65}")
print(f"FALSE POSITIVES per clasă:")
print(f"{'─'*65}")
for label in VALID_LABELS:
    fp         = sum(len(v) for (t, p), v in errors.items() if p == label)
    total_pred = sum(1 for p in preds if p.get(PRED_FIELD) == label and not p.get("parse_failed"))
    print(f"  {label:<28} FP={fp:>2}/{total_pred:<3} ", end="")
    breakdown = Counter({t: len(v) for (t, p), v in errors.items() if p == label})
    for true_was, cnt in breakdown.most_common(3):
        print(f"  ←{true_was}:{cnt}", end="")
    print()

# --- 5. DETALII ERORI (conversation_id + reasoning) ---
print(f"\n{'='*90}")
print(f"DETALII ERORI PE BEST CONFIG — {best_name}")
print(f"{'='*90}")
for (true, pred), lst in sorted(errors.items(), key=lambda x: -len(x[1])):
    print(f"\n  {true} → {pred}  ({len(lst)})")
    print(f"  {'─'*60}")
    for p in lst:
        conv_id  = p.get(CONV_FIELD, "?")
        conf     = p.get("confidence", "?")
        reason   = p.get("reasoning", "—")
        print(f"    [{conv_id}] conf={conf}")
        print(f"    {reason}")

# --- 6. CANDIDAȚI FEW-SHOT (clasificate corect cu high confidence) ---
print(f"\n{'='*90}")
print(f"CANDIDAȚI FEW-SHOT (corect + confidence=high)")
print(f"{'='*90}")
for label in VALID_LABELS:
    candidates = [p for p in correct_by_class[label]
                  if str(p.get("confidence", "")).lower() == "high"]
    print(f"\n  {label}  ({len(candidates)} cu high confidence, {len(correct_by_class[label])} total corecte)")
    for p in candidates[:3]:
        print(f"    [{p.get(CONV_FIELD,'?')}]  {p.get('reasoning','—')}")
    if not candidates and correct_by_class[label]:
        # fallback: arată primele 3 corecte indiferent de confidence
        print(f"    (niciun high — primele 3 corecte indiferent de confidence:)")
        for p in correct_by_class[label][:3]:
            print(f"    [{p.get(CONV_FIELD,'?')}] conf={p.get('confidence','?')}  {p.get('reasoning','—')}")

# --- 7. GHID CONSTRUCȚIE v4 ---
print(f"\n{'='*90}")
print(f"GHID EXEMPLE FEW-SHOT v4")
print(f"{'='*90}")
top5 = sorted(errors.items(), key=lambda x: -len(x[1]))[:5]
print("\nTop confuzii de acoperit cu exemple:")
for (true, pred), lst in top5:
    print(f"  ⚡ {true:<25} → {pred:<22}  ({len(lst)} erori)")
print(f"""
Structura recomandată:
  • 5 exemple pozitive — câte unul per etichetă
  • 2 exemple negative — pentru cele mai frecvente false positive-uri

Format JSON  →  configs/few_shot_examples_final_status.json
{{
  "examples": [
    {{
      "conversation": [
        {{"role": "user", "text": "..."}},
        {{"role": "assistant", "text": "..."}}
      ],
      "final_status": "rezolvata",
      "reasoning": "Botul a blocat cardul; utilizatorul doar așteaptă confirmarea."
    }}
  ]
}}
""")

Experimente găsite: 25
  • fst_aya_expanse_8b__en__v1
  • fst_aya_expanse_8b__en__v2
  • fst_aya_expanse_8b__ro__v1
  • fst_aya_expanse_8b__ro__v2
  • fst_aya_expanse_8b__ro__v3
  • fst_gemini_2.5_flash__en__v1
  • fst_gemini_2.5_flash__en__v2
  • fst_gemini_2.5_flash__en__v3
  • fst_gemini_2.5_flash__ro__v1
  • fst_gemini_2.5_flash__ro__v2
  • fst_gemini_2.5_flash__ro__v3
  • fst_openai_o3__en__v1
  • fst_openai_o3__en__v2
  • fst_openai_o3__en__v3
  • fst_openai_o3__ro__v1
  • fst_openai_o3__ro__v2
  • fst_openai_o3__ro__v3
  • fst_roberta_encoder__en__v1
  • fst_roberta_encoder__ro__v1
  • fst_roberta_encoder__ro__v2
  • fst_roberta_encoder__ro__v3
  • fst_rollama2_7b__en__v1
  • fst_rollama2_7b__en__v2
  • fst_rollama2_7b__ro__v1
  • fst_rollama2_7b__ro__v2

Chei metrics: ['accuracy', 'macro_f1', 'weighted_f1', 'kappa', 'n_valid']
Kappa key: 'kappa'

SUMAR METRICI — toate versiunile
Experiment                                            Acc  MacF1  WgtF1   Kappa
--------------------

In [10]:
# [9] Analiză erori per experiment
# ─────────────────────────────────────────────────────────────────────────
import json
from pathlib import Path
from collections import defaultdict

# Încarcă experimentul curent sau selectează unul
# Opțiunea 1: folosește `results` din memorie (dacă tocmai ai rulat celula [5])
# Opțiunea 2: încarcă din fișier
LOAD_FROM_FILE = True  # True = încarcă din fișier, False = folosește `results` din memorie
EXP_TO_LOAD = "fst_openai_o3__ro__v1"      # None = cel mai recent, sau "fst_openai_o3__ro__v1"

if LOAD_FROM_FILE:
    if EXP_TO_LOAD:
        exp_file = OUTPUT_DIR / f"exp_{EXP_TO_LOAD}.json"
    else:
        exp_files = sorted(OUTPUT_DIR.glob("exp_fst_*.json"))
        exp_file = exp_files[-1] if exp_files else None

    if exp_file and exp_file.exists():
        with open(exp_file, encoding="utf-8") as f:
            exp_data = json.load(f)
        results = exp_data.get("results", exp_data.get("predictions", []))
        print(f"✓ Încărcat: {exp_file.name}")
    else:
        print(f"✗ Nu s-a găsit fișierul")
else:
    print(f"✓ Folosesc `results` din memorie ({len(results)} predicții)")

# ─── Clasifică erorile ────────────────────────────────────────────────────
errors = []
correct = 0
fails = 0

for r in results:
    if r["parse_failed"]:
        fails += 1
        continue

    true_status = r["dataset_status"]
    pred_status = r["predicted_status"]

    if pred_status == true_status:
        correct += 1
    else:
        errors.append(r)

print(f"\n{'═' * 80}")
print(f"SUMAR: {correct} corecte, {len(errors)} greșeli, {fails} parse failures")
print(f"{'═' * 80}\n")

# ─── Grupează erorile per tip de confuzie ──────────────────────────────────
confusion = defaultdict(list)
for e in errors:
    key = f"{e['dataset_status']} → {e['predicted_status']}"
    confusion[key].append(e)

# Sortează după frecvență
sorted_confusions = sorted(confusion.items(), key=lambda x: -len(x[1]))

print(f"{'─' * 80}")
print(f"ERORI GRUPATE PER CONFUZIE ({len(errors)} total)")
print(f"{'─' * 80}\n")

for conf_type, conv_list in sorted_confusions:
    print(f"▸ {conf_type}  ({len(conv_list)} greșeli)")
    print(f"  {'·' * 60}")
    for c in conv_list:
        conv_id = c["conversation_id"]
        reasoning = c.get("reasoning", "—")
        confidence = c.get("confidence", "—")
        print(f"  {conv_id:<25} conf={confidence:<6}  {reasoning}")
    print()

# ─── Afișează conversațiile greșite cu detalii ─────────────────────────────
print(f"\n{'═' * 80}")
print(f"DETALII CONVERSAȚII GREȘITE")
print(f"{'═' * 80}\n")

# Încarcă dataset-ul pentru a afișa turnurile
with open(DATASET_PATH, encoding="utf-8") as f:
    all_convs = {c["conversation_id"]: c for c in json.load(f)["conversations"]}

for i, e in enumerate(errors):
    conv_id = e["conversation_id"]
    conv_data = all_convs.get(conv_id, {})
    turns = conv_data.get("turns", [])

    print(f"{'─' * 80}")
    print(f"[{i+1}/{len(errors)}]  {conv_id}")
    print(f"  Real: {e['dataset_status']}  |  Prezis: {e['predicted_status']}  |  Confidence: {e.get('confidence', '—')}")
    print(f"  Reasoning: {e.get('reasoning', '—')}")
    print()

    # Afișează ultimele turnuri (cele relevante pentru decizie)
    display_turns = turns[-6:] if len(turns) > 6 else turns
    if len(turns) > 6:
        print(f"  ... ({len(turns) - 6} turnuri anterioare omise)")
    for t in display_turns:
        role = t["role"].upper()
        text = t["text"][:200] + "..." if len(t["text"]) > 200 else t["text"]
        print(f"  {role}: {text}")
    print()

print(f"{'═' * 80}")
print(f"Total erori: {len(errors)} / {len(results)} ({100*len(errors)/len(results):.1f}%)")
print(f"{'═' * 80}")

✓ Încărcat: exp_fst_openai_o3__ro__v1.json

════════════════════════════════════════════════════════════════════════════════
SUMAR: 131 corecte, 49 greșeli, 0 parse failures
════════════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────────────
ERORI GRUPATE PER CONFUZIE (49 total)
────────────────────────────────────────────────────────────────────────────────

▸ rezolvata → partial_rezolvata  (14 greșeli)
  ····························································
  conv_simple_0029          conf=high    Raportarea a fost doar înregistrată, iar utilizatorul trebuie să aștepte pașii următori de la bancă.
  conv_simple_0030          conf=high    Raportul a fost doar înregistrat, iar utilizatorul trebuie să aștepte verificarea și apelul ulterior.
  conv_simple_0031          conf=medium  Raportul a fost doar înregistrat; soluția finală urmează după investigație, deci rezolvare incompletă.
  con

In [19]:
# =============================================================
# ANALIZA ERORILOR — Final Status v2
# Rulează această celulă după ce toate experimentele v2 sunt finalizate.
# Compatibilă cu fișiere care au:
#   - results SAU predictions
#   - dataset_status SAU dataset_label
#   - experiment_name SAU experiment.name
# =============================================================

import json, os, glob
from collections import Counter, defaultdict
import numpy as np

# --- CONFIG ---
OUTPUTS_DIR = "outputs_final_status"
VERSION = "v2"
PREFIX = "exp_fst_"  # ajustează dacă prefixul diferă

# Field names posibile
PRED_FIELD = "predicted_status"
LABEL_FIELD_CANDIDATES = ["dataset_status", "dataset_label"]
CONV_ID_FIELD = "conversation_id"

VALID_LABELS = [
    "rezolvata",
    "partial_rezolvata",
    "redirectionata",
    "intrerupta",
    "nerezolvata"
]

# =============================================================
# HELPER FUNCTIONS
# =============================================================

def get_experiment_name(data, fpath):
    """
    Extrage numele experimentului indiferent de structura JSON-ului.
    Suportă:
      - experiment_name
      - experiment.name
      - fallback: numele fișierului
    """
    if "experiment_name" in data:
        return data["experiment_name"]

    if isinstance(data.get("experiment"), dict):
        return data["experiment"].get("name", os.path.basename(fpath))

    return os.path.basename(fpath)


def get_predictions(data):
    """
    Extrage lista de predicții indiferent dacă fișierul folosește:
      - results
      - predictions
    """
    if "results" in data:
        return data["results"]

    if "predictions" in data:
        return data["predictions"]

    raise KeyError(
        f"Nu am găsit nici cheia 'results', nici cheia 'predictions'. "
        f"Chei disponibile: {list(data.keys())}"
    )


def get_label_field(preds):
    """
    Detectează automat câmpul de ground truth:
      - dataset_status
      - dataset_label
    """
    if not preds:
        raise ValueError("Lista de predicții este goală.")

    for field in LABEL_FIELD_CANDIDATES:
        if field in preds[0]:
            return field

    raise KeyError(
        f"Nu am găsit niciun câmp de label dintre {LABEL_FIELD_CANDIDATES}. "
        f"Cheile primului rezultat sunt: {list(preds[0].keys())}"
    )


def get_true_label(p, label_field):
    """
    Returnează label-ul real.
    """
    return p.get(label_field, "MISSING_LABEL")


def get_pred_label(p):
    """
    Returnează predicția modelului.
    Dacă parse_failed=True, marchează PARSE_FAIL.
    """
    if p.get("parse_failed", False):
        return "PARSE_FAIL"

    return p.get(PRED_FIELD, "MISSING_PRED")


# =============================================================
# 1. ÎNCARCĂ TOATE EXPERIMENTELE v2
# =============================================================

pattern = os.path.join(OUTPUTS_DIR, f"{PREFIX}*_{VERSION}.json")
files = sorted(glob.glob(pattern))

experiments = {}

for fpath in files:
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    exp_name = get_experiment_name(data, fpath)
    experiments[exp_name] = data

print(f"Fișiere căutate după pattern:")
print(f"  {pattern}")

print(f"\nExperimente v2 găsite: {len(experiments)}")

for name in experiments:
    print(f"  • {name}")

if not experiments:
    raise FileNotFoundError(
        f"Nu am găsit niciun fișier pentru pattern-ul: {pattern}\n"
        f"Verifică OUTPUTS_DIR, PREFIX și VERSION."
    )

# =============================================================
# 2. TABEL SUMAR METRICI
# =============================================================

print("\n" + "=" * 90)
print("METRICI SUMAR — toate experimentele v2")
print("=" * 90)
print(f"{'Experiment':<45} {'Acc':>6} {'MacF1':>6} {'WgtF1':>6} {'Kappa':>6} {'Parse%':>7}")
print("-" * 90)

best_exp_name = None
best_macro_f1 = -1

for name, data in experiments.items():
    m = data.get("metrics", {})
    preds = get_predictions(data)

    parse_fails = sum(1 for p in preds if p.get("parse_failed", False))
    parse_pct = (parse_fails / len(preds) * 100) if preds else 0

    acc = m.get("accuracy", 0)
    macro_f1 = m.get("macro_f1", 0)
    weighted_f1 = m.get("weighted_f1", 0)
    kappa = m.get("cohen_kappa", 0)

    print(
        f"{name:<45} "
        f"{acc:>6.3f} "
        f"{macro_f1:>6.3f} "
        f"{weighted_f1:>6.3f} "
        f"{kappa:>6.3f} "
        f"{parse_pct:>6.1f}%"
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_exp_name = name

print("-" * 90)
print(f"\n★ Best Macro-F1: {best_exp_name} ({best_macro_f1:.3f})")

# =============================================================
# 3. CONFUSION MATRIX — best experiment
# =============================================================

best_data = experiments[best_exp_name]
preds = get_predictions(best_data)
LABEL_FIELD = get_label_field(preds)

print(f"\nCâmp ground truth detectat automat: {LABEL_FIELD}")
print(f"Câmp predicție folosit: {PRED_FIELD}")

print(f"\n{'=' * 90}")
print(f"MATRICE DE CONFUZIE — {best_exp_name}")
print(f"{'=' * 90}")

cm = defaultdict(lambda: defaultdict(int))

for p in preds:
    true = get_true_label(p, LABEL_FIELD)
    pred = get_pred_label(p)
    cm[true][pred] += 1

extra_pred_labels = sorted(
    {
        get_pred_label(p)
        for p in preds
        if get_pred_label(p) not in VALID_LABELS
    }
)

all_labels = VALID_LABELS + extra_pred_labels

header = f"{'True \\ Pred':<22}" + "".join(f"{l:>16}" for l in all_labels) + f"{'Total':>8}"
print(header)
print("-" * len(header))

for true_label in VALID_LABELS:
    row = f"{true_label:<22}"
    total = 0

    for pred_label in all_labels:
        val = cm[true_label].get(pred_label, 0)
        total += val
        marker = " ✓" if true_label == pred_label else ""
        row += f"{val:>14}{marker:>2}"

    row += f"{total:>8}"
    print(row)

# Afișează și eventuale label-uri reale necunoscute
unknown_true_labels = sorted(
    {
        get_true_label(p, LABEL_FIELD)
        for p in preds
        if get_true_label(p, LABEL_FIELD) not in VALID_LABELS
    }
)

if unknown_true_labels:
    print("\nAtenție: există label-uri reale care nu sunt în VALID_LABELS:")
    for label in unknown_true_labels:
        print(f"  - {label}")

# =============================================================
# 4. ANALIZA ERORILOR PER CLASĂ
# =============================================================

print(f"\n{'=' * 90}")
print(f"ANALIZA ERORILOR PER CLASĂ — {best_exp_name}")
print(f"{'=' * 90}")

errors_by_type = defaultdict(list)

for p in preds:
    true = get_true_label(p, LABEL_FIELD)
    pred = get_pred_label(p)

    if true != pred:
        errors_by_type[(true, pred)].append(p)

for label in VALID_LABELS:
    fn_count = sum(
        len(v)
        for (true_label, pred_label), v in errors_by_type.items()
        if true_label == label
    )

    fp_count = sum(
        len(v)
        for (true_label, pred_label), v in errors_by_type.items()
        if pred_label == label
    )

    total_true = sum(
        1
        for p in preds
        if get_true_label(p, LABEL_FIELD) == label
    )

    total_pred = sum(
        1
        for p in preds
        if get_pred_label(p) == label
    )

    correct = total_true - fn_count

    print(f"\n--- {label} ---")
    print(f"  Ground truth: {total_true} | Predicted: {total_pred} | Correct: {correct}")
    print(f"  False Negatives — era {label}, prezis altceva: {fn_count}")

    if fn_count > 0:
        fn_details = Counter()

        for (true_label, pred_label), error_list in errors_by_type.items():
            if true_label == label:
                fn_details[pred_label] += len(error_list)

        for pred_as, count in fn_details.most_common():
            print(f"    → prezis ca {pred_as}: {count}")

    print(f"  False Positives — era altceva, prezis ca {label}: {fp_count}")

    if fp_count > 0:
        fp_details = Counter()

        for (true_label, pred_label), error_list in errors_by_type.items():
            if pred_label == label:
                fp_details[true_label] += len(error_list)

        for true_was, count in fp_details.most_common():
            print(f"    → era de fapt {true_was}: {count}")

# =============================================================
# 5. TOP CONFUZII
# =============================================================

print(f"\n{'=' * 90}")
print("TOP CONFUZII — perechi true → pred, ordonate după frecvență")
print(f"{'=' * 90}")

confusion_pairs = Counter()

for (true_label, pred_label), error_list in errors_by_type.items():
    confusion_pairs[(true_label, pred_label)] = len(error_list)

if not confusion_pairs:
    print("Nu există erori în best experiment.")
else:
    for (true_label, pred_label), count in confusion_pairs.most_common(15):
        bar = "█" * min(count, 50)
        print(f"  {true_label:<22} → {pred_label:<22} : {count:>3}  {bar}")

# =============================================================
# 6. TOTAL ERORI
# =============================================================

total_errors = sum(len(v) for v in errors_by_type.values())
total_preds = len(preds)

print(f"\nTotal erori: {total_errors}/{total_preds} ({total_errors / total_preds * 100:.1f}%)")
print(f"Acuratețe calculată din rezultate: {(total_preds - total_errors) / total_preds * 100:.1f}%")

# =============================================================
# 7. DETALII ERORI
# =============================================================

print(f"\n{'=' * 90}")
print(f"DETALII ERORI — {best_exp_name}")
print("(conversation_id, true, predicted, confidence, reasoning)")
print(f"{'=' * 90}")

if not errors_by_type:
    print("Nu există erori de afișat.")
else:
    for (true_label, pred_label), error_list in sorted(
        errors_by_type.items(),
        key=lambda x: -len(x[1])
    ):
        print(f"\n{'─' * 70}")
        print(f"  {true_label} → {pred_label}  ({len(error_list)} erori)")
        print(f"{'─' * 70}")

        for p in error_list:
            conv_id = p.get(CONV_ID_FIELD, "?")
            conf = p.get("confidence", "?")
            reason = p.get("reasoning", "—")

            print(f"  [{conv_id}] conf={conf}")
            print(f"    reasoning: {reason}")
            print()

# =============================================================
# 8. COMPARAȚIE RO vs EN PER MODEL
# =============================================================

print(f"\n{'=' * 90}")
print("COMPARAȚIE RO vs EN per model")
print(f"{'=' * 90}")

models_seen = defaultdict(dict)

for name, data in experiments.items():
    m = data.get("metrics", {})

    # Preferă câmpurile explicite din JSON, dacă există
    model = data.get("model")
    lang = data.get("language")

    # Fallback pe parsare din nume:
    # exemple:
    #   fst_gemini_2.5_flash__ro__v2
    #   gemini_2.5_flash_ro_v2
    if not model or not lang:
        clean_name = name

        if clean_name.startswith("fst_"):
            clean_name = clean_name[len("fst_"):]

        if "__" in clean_name:
            parts = clean_name.split("__")
            if len(parts) >= 3:
                model = parts[0]
                lang = parts[1]
        else:
            parts = clean_name.rsplit("_", 2)
            if len(parts) >= 3:
                model = "_".join(parts[:-2])
                lang = parts[-2]

    if not model or not lang:
        continue

    models_seen[model][lang] = {
        "accuracy": m.get("accuracy", 0),
        "macro_f1": m.get("macro_f1", 0),
        "kappa": m.get("cohen_kappa", 0),
    }

if models_seen:
    print(f"{'Model':<35} {'RO Acc':>7} {'EN Acc':>7} {'RO F1':>7} {'EN F1':>7} {'Δ F1':>7}")
    print("-" * 80)

    for model, langs in sorted(models_seen.items()):
        ro = langs.get("ro", {})
        en = langs.get("en", {})

        ro_f1 = ro.get("macro_f1", 0)
        en_f1 = en.get("macro_f1", 0)
        delta = ro_f1 - en_f1

        print(
            f"{model:<35} "
            f"{ro.get('accuracy', 0):>7.3f} "
            f"{en.get('accuracy', 0):>7.3f} "
            f"{ro_f1:>7.3f} "
            f"{en_f1:>7.3f} "
            f"{delta:>+7.3f}"
        )
else:
    print("Nu am putut extrage model/lang pentru comparația RO vs EN.")

# =============================================================
# 9. META-PROMPT HELPER
# =============================================================

print(f"\n{'=' * 90}")
print("REZUMAT ERORI PENTRU META-PROMPT — copy-paste")
print(f"{'=' * 90}")

label_distribution = {
    label: sum(
        1
        for p in preds
        if get_true_label(p, LABEL_FIELD) == label
    )
    for label in VALID_LABELS
}

print(f"""
Task: clasificare status final al conversației — 5 clase
Dataset: {total_preds} conversații bancare în română

Distribuție:
  - rezolvata: {label_distribution['rezolvata']}
  - partial_rezolvata: {label_distribution['partial_rezolvata']}
  - redirectionata: {label_distribution['redirectionata']}
  - intrerupta: {label_distribution['intrerupta']}
  - nerezolvata: {label_distribution['nerezolvata']}

Best model pe v2: {best_exp_name}
Macro-F1: {best_macro_f1:.3f}

Erori sistematice observate pe best config:
""")

if confusion_pairs:
    for (true_label, pred_label), count in confusion_pairs.most_common(10):
        pct = count / total_preds * 100
        print(f"  - {true_label} clasificat greșit ca {pred_label}: {count} cazuri ({pct:.1f}%)")
else:
    print("  - Nu există erori în best config.")

print(f"""

Total erori: {total_errors}/{total_preds} ({total_errors / total_preds * 100:.1f}%)

Constrângeri de format:
  - Păstrează exact aceste câmpuri JSON: final_status, confidence, reasoning
  - Păstrează exact aceste etichete: {', '.join(VALID_LABELS)}
  - Păstrează template-ul Jinja2 pentru injectarea conversației
  - Promptul trebuie să funcționeze atât în română cât și în engleză
""")

Fișiere căutate după pattern:
  outputs_final_status\exp_fst_*_v2.json

Experimente v2 găsite: 9
  • fst_aya_expanse_8b__en__v2
  • fst_aya_expanse_8b__ro__v2
  • fst_gemini_2.5_flash__en__v2
  • fst_gemini_2.5_flash__ro__v2
  • fst_openai_o3__en__v2
  • fst_openai_o3__ro__v2
  • fst_roberta_encoder__ro__v2
  • fst_rollama2_7b__en__v2
  • fst_rollama2_7b__ro__v2

METRICI SUMAR — toate experimentele v2
Experiment                                       Acc  MacF1  WgtF1  Kappa  Parse%
------------------------------------------------------------------------------------------
fst_aya_expanse_8b__en__v2                     0.733  0.614  0.702  0.000    0.0%
fst_aya_expanse_8b__ro__v2                     0.717  0.607  0.684  0.000    0.0%
fst_gemini_2.5_flash__en__v2                   0.839  0.774  0.835  0.000    0.0%
fst_gemini_2.5_flash__ro__v2                   0.817  0.763  0.816  0.000    0.0%
fst_openai_o3__en__v2                          0.821  0.746  0.816  0.000    0.6%
fst_openai_o